# ResNet50 + Contrastive Regularization (train from scratch)
This notebook train from scratch ResNet50 with a contrastive regularizer.

Sections:
- 1) Setup and Imports
- 2) Dataset and Dataloaders
- 3) Model and Loss
- 4) Training Loop
- 5) Evaluation and Plots

Tips: Set `lambda_contrastive=0.0` to run CE-only baselines.


In [2]:
# 1) Setup and Imports
# Basic utilities and HDF5 visualization helper
import h5py
from dl_utils.utils.utils import viz_h5_structure

In [3]:
with h5py.File('../../datasets/imagenet_v5_rot_10m_fix_vector-hierarchy_metadata.h5', 'r') as f:
    viz_h5_structure(f)

'Group': imagenet
  'Dataset': data; Shape: (10173208, 256, 256, 3); dtype: uint8
  'Dataset': labels; Shape: (10173208,); dtype: uint8
  'Dataset': normalized_primitive_uc_vector_a; Shape: (10173208, 2); dtype: float32
  'Dataset': normalized_primitive_uc_vector_b; Shape: (10173208, 2); dtype: float32
  'Dataset': normalized_translation_start_point; Shape: (10173208, 2); dtype: float32
  'Dataset': primitive_uc_vector_a; Shape: (10173208, 2); dtype: int32
  'Dataset': primitive_uc_vector_b; Shape: (10173208, 2); dtype: int32
  'Dataset': rotation_angle; Shape: (10173208,); dtype: uint8
  'Dataset': shape; Shape: (10173208,); dtype: uint8
  'Dataset': shape_labels_refined; Shape: (10173208,); dtype: uint8
  'Dataset': symmetry_features; Shape: (10173208, 6); dtype: uint8
  'Dataset': symmetry_vector; Shape: (10173208, 13); dtype: float32
  'Dataset': translation_start_point; Shape: (10173208, 2); dtype: int32
  'Dataset': translation_uc_vector_a; Shape: (10173208, 2); dtype: int32
  'D

In [4]:
# Additional imports for training and visualization
# Model builder, trainer, and the contrastive-regularized loss wrapper
import os
import json
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import transforms

from dl_utils.utils.dataset import hdf5_dataset, split_train_valid, viz_dataloader
from dl_utils.utils.utils import list_to_dict
from dl_utils.training.build_model import resnet50_
from dl_utils.training.trainer import Trainer, accuracy
from dl_utils.training.contrastive_loss import ContrastiveRegularizedLoss
import wandb

# Symmetry classes and helper to map label -> index
symmetry_classes = ['p1', 'p2', 'pm', 'pg', 'cm', 'pmm', 'pmg', 'pgg', 'cmm', 'p4', 'p4m', 'p4g', 'p3', 'p3m1', 'p31m', 'p6', 'p6m']
label_converter = list_to_dict(symmetry_classes)
n_classes = len(symmetry_classes)


/mnt/scratch/home/yichen/anaconda3/envs/symmetry/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
# 2) Config: dataset paths, training hyperparameters, and contrastive settings
task_name = "ResNet50-fine_tune"
ds_path_info = {
    'imagenet': '../../datasets/imagenet_v5_rot_10m_fix_vector-hierarchy_metadata.h5',
    'atom': '../../datasets/atom_v5_rot_1m_fix_vector.h5',
    'noise': '../../datasets/noise_v5_rot_1m_fix_vector.h5',
    'viz_dataloader': False,  # set True to visualize few batches
}

training_specs = {
    'ds_size': 10_000_000,
    'batch_size': 3000,
    'num_workers': 24,
    'learning_rate': 1e-3,
    'training_image_count': 2_000_000,
    'validation_times': 100,
    'efficient_print': True,
    'model_path': '../../models/ResNet50/',
    'folder_name': 'default',
    'device_ids': [4, 5, 6, 7, 8, 9],
    'epoch_start': 0,
    'contrastive_warmup_epochs': 5,
    'lambda_contrastive_final': 1e-20,
}

# W&B (optional)
wandb_specs = {}

metadata_key = 'symmetry_vector'
metadata_weights = [1.0] * 6 + [1.0 / 7.0] + [0.5] * 6
lambda_contrastive_final = training_specs['lambda_contrastive_final']
contrastive_warmup_epochs = training_specs['contrastive_warmup_epochs']
# lambda_contrastive = 1e-20
pos_threshold = 0.75
neg_threshold = 1.6
margin = 1.0
metadata_distance = 'l2'
feature_layer = 'avgpool'
feature_norm = True


In [6]:
# 2b) Contrastive weight schedule

def contrastive_lambda_schedule(epoch_idx: int):
    '''Linear warmup for lambda_contrastive over the first warmup epochs.'''
    if contrastive_warmup_epochs <= 0:
        return lambda_contrastive_final
    clamped_epoch = max(0, epoch_idx)
    ratio = min(1.0, clamped_epoch / contrastive_warmup_epochs)
    return lambda_contrastive_final * ratio

lambda_contrastive = contrastive_lambda_schedule(0)


In [7]:
# 2) Dataset and Dataloaders
# Imagenet with metadata for training
imagenet_all = hdf5_dataset(
    ds_path_info['imagenet'],
    folder='imagenet',
    transform=transforms.ToTensor(),
    metadata_keys=metadata_key,
)
ratio = training_specs['ds_size'] * (1 / 0.8) / len(imagenet_all)
imagenet_sub, _ = split_train_valid(imagenet_all, ratio, seed=42)
train_ds, valid_ds = split_train_valid(imagenet_sub, 0.8, seed=42)

train_dl = DataLoader(
    train_ds,
    batch_size=training_specs['batch_size'],
    shuffle=True,
    num_workers=training_specs['num_workers'],
)
valid_dl = DataLoader(
    valid_ds,
    batch_size=training_specs['batch_size'],
    shuffle=False,
    num_workers=training_specs['num_workers'],
)

# Noise CV
noise_ds_all = hdf5_dataset(ds_path_info['noise'], folder='noise', transform=transforms.ToTensor())
ratio_noise = np.min((training_specs['ds_size'] / len(noise_ds_all), 1))
noise_ds, _ = split_train_valid(noise_ds_all, ratio_noise, seed=42)
noise_dl = DataLoader(
    noise_ds,
    batch_size=training_specs['batch_size'],
    shuffle=False,
    num_workers=training_specs['num_workers'],
)

# Atom CV
atom_ds_all = hdf5_dataset(ds_path_info['atom'], folder='atom', transform=transforms.ToTensor())
ratio_atom = np.min((training_specs['ds_size'] / len(atom_ds_all), 1))
atom_ds, _ = split_train_valid(atom_ds_all, ratio_atom, seed=42)
atom_dl = DataLoader(
    atom_ds,
    batch_size=training_specs['batch_size'],
    shuffle=False,
    num_workers=training_specs['num_workers'],
)

if ds_path_info['viz_dataloader']:
    viz_dataloader(valid_dl, label_converter=label_converter, title='imagenet - valid')
    viz_dataloader(noise_dl, label_converter=label_converter, title='noise - valid')
    viz_dataloader(atom_dl, label_converter=label_converter, title='atom - valid')


In [8]:
# 3) Model: ResNet50 backbone (with optional DataParallel)
device = torch.device(f"cuda:{training_specs['device_ids'][0]}" if torch.cuda.is_available() else "cpu")

model = resnet50_(in_channels=3, n_classes=n_classes)
model.load_state_dict(torch.load('../../models/ResNet50/03132025-ResNet50-benchmark-10m/epoch_23.pth', weights_only=True, map_location=torch.device('cpu')))

if len(training_specs['device_ids']) > 1:
    model = torch.nn.DataParallel(model, device_ids=training_specs['device_ids'])
model = model.to(device)

In [9]:
# 3b) Loss, optimizer, scheduler, and metrics
lr = training_specs['learning_rate']
epoch_start = training_specs.get('epoch_start', 0)
# epochs = training_specs['training_image_count'] // len(train_ds) - epoch_start
epochs = 100
valid_per_epochs = int(np.max((1, epochs / training_specs['validation_times'])))
early_stopping_patience = np.max((5, valid_per_epochs + 2))
efficient_print = training_specs['efficient_print']

base_ce = nn.CrossEntropyLoss()
loss_func = ContrastiveRegularizedLoss(
    base_criterion=base_ce,
    lambda_contrastive=lambda_contrastive,
    feature_layer=feature_layer,
    pos_threshold=pos_threshold,
    neg_threshold=neg_threshold,
    margin=margin,
    feature_norm=feature_norm,
    metadata_key=metadata_key,
    metadata_distance=metadata_distance,
    metadata_weights=metadata_weights,
)

optimizer = torch.optim.Adam(model.parameters(), lr=lr)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    epochs=epochs,
    max_lr=lr,
    steps_per_epoch=len(train_dl),
)
metrics = [accuracy]


In [10]:
# 3c) Weights & Biases setup (optional)
NAME = task_name + '-dstaset_size=' + str(training_specs['ds_size'])
if wandb_specs:
    wandb.login()
    wandb.init(
        project=wandb_specs['project'], entity=wandb_specs['entity'],
        name=NAME, id=NAME, group=wandb_specs['group'],
        save_code=wandb_specs['save_code'], config=wandb_specs['config'], resume=wandb_specs['resume']
    )
    training_specs['wandb_record'] = True
else:
    training_specs['wandb_record'] = False

# 4) Training Loop and Validation
Runs the Trainer and logs per-epoch metrics including base CE and contrastive components (if enabled).


In [ ]:
folder_name = training_specs['folder_name'] if training_specs['folder_name'] != 'default' else NAME
model_dir = os.path.join(training_specs['model_path'], folder_name)
os.makedirs(model_dir, exist_ok=True)

trainer = Trainer(
    model=model,
    loss_func=loss_func,
    optimizer=optimizer,
    metrics=metrics,
    scheduler=scheduler,
    device=device,
    save_per_epochs=valid_per_epochs,
    model_path=model_dir + '/',
    early_stopping_patience=early_stopping_patience,
    efficient_print=efficient_print,
)

history = trainer.train(
    train_dl=train_dl,
    epochs=epochs,
    epoch_start=epoch_start,
    valid_per_epochs=valid_per_epochs,
    valid_dl_list=[valid_dl, noise_dl, atom_dl],
    valid_dl_names=['', 'noise', 'atom'],
    wandb_record=training_specs['wandb_record'],
    contrastive_lambda_schedule=contrastive_lambda_schedule,
)


[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99]
Epoch: 1/100


Train: 100%|██████████| 2713/2713 [1:21:48<00:00,  1.81s/it]


train_loss: 0.0338, train_accuracy: 98.78%, train_loss_base: 0.0338, train_loss_contrastive: 0.0000e+00, train_lambda_contrastive: 0.0000e+00


Valid: 100%|██████████| 679/679 [11:38<00:00,  1.03s/it]


valid_loss: 0.0369, valid_accuracy: 98.68%, valid_loss_base: 0.0369, valid_loss_contrastive: 0.0000e+00
Model saved at epoch 0
Saved new best model at epoch 0 with valid dataset


...

Valid: 100%|██████████| 334/334 [05:04<00:00,  1.10it/s]


noise_loss: 2.8332, noise_accuracy: 5.88%, noise_loss_base: 2.8332, noise_loss_contrastive: 0.0000e+00


Valid: 100%|██████████| 334/334 [05:09<00:00,  1.08it/s]


atom_loss: 2.8332, atom_accuracy: 5.88%, atom_loss_base: 2.8332, atom_loss_contrastive: 0.0000e+00
Model saved at epoch 24
Epoch: 25/100


Train:  80%|███████▉  | 2161/2713 [47:45<11:55,  1.30s/it] 